# System Interaction Analysis (Five-Layer Deep Dive)

This notebook captures a full architecture analysis focused on how components interact across system boundaries, contracts, execution flows, and operations.

## Scope and assumptions

- Canonical intent: notebook-first platform reference for ML engineering workflows.
- Infrastructure direction: OpenTofu generated via Terranix from both flake and devenv paths.
- Runtime landscapes: local replica, Lambda.ai (Slurm), AWS (Kubernetes for non-Lambda.ai services).
- Core cross-cutting obligations: security, lineage, experiment/model traceability, reproducibility, cost attribution.

## Layer 1 — System context

### Primary actors

- ML Engineer: authors notebooks, triggers runs, evaluates outputs, promotes model versions.
- Platform Engineer: owns infrastructure definitions, environment parity, orchestration reliability.
- Reviewer / Governance Owner: validates security, lineage, deployment approvals, and auditability.

### External systems

- Git repository: immutable source for notebook revisions.
- MLflow: run tracking, model metadata, run-level visibility.
- Storage + metadata systems: PostgreSQL backend + S3-compatible artifacts.
- Compute orchestration: local execution, Slurm (Lambda.ai), Kubernetes (AWS non-Lambda workloads).

### Boundary statement

The platform controls orchestration contracts, metadata continuity, and execution routing. External platforms provide compute/storage primitives and are integrated through explicit contracts rather than hidden coupling.

## Layer 2 — Component architecture

| Component | Responsibility | Inputs | Outputs | Upstream dependencies |
| --- | --- | --- | --- | --- |
| Notebook intake | Validate immutable notebook revision and request shape | NotebookExecutionRequest + Git revision | Normalized job spec | Git, contract validators |
| Execution orchestrator | Route job spec to local/slurm/k8s path | Normalized job spec + runtime config | Completed local run or submission payload | webui_contracts, execution_backends |
| Vertical slice runtime | Execute local train/package/deploy/predict path | Work dir + tracking URI + synthetic or real dataset | MLflow run, artifact metadata, deployment record, prediction logs | mlflow, sklearn, storage |
| MLflow parity config | Resolve backend/artifact/tracking settings | Environment and parity defaults | Runtime env map + startup commands | Postgres/S3/MinIO topology |
| Infra config generation | Emit OpenTofu JSON config via Terranix from flake/devenv | Nix expressions + environment profile | OpenTofu `*.tf.json` artifacts | flake.nix, devenv.nix, Terranix |

### Ownership split

- Contract and orchestration modules own behavior contracts and routing.
- Vertical slice owns first executable architecture proof.
- Infrastructure generation layer owns reproducible infra configuration artifacts and environment parity.

## Layer 3 — Interaction contracts

### Contract chain

1. Web UI emits immutable execution request (`NotebookExecutionRequest`).
2. Intake and request normalization produce canonical job specification.
3. Orchestrator maps spec to backend-specific submission or local execution.
4. Execution emits traceability records into MLflow/artifacts/deployment/prediction logs.
5. Run visibility links return to engineers and operations surfaces.

### Required invariants

- Every execution references immutable notebook revision.
- Every prediction can be traced to model version and originating run.
- Every deployment record references promoted model version.
- Every environment path emits equivalent core lineage fields.
- Infrastructure generation paths (flake and devenv) converge on equivalent OpenTofu JSON semantics.

### Failure semantics (current expectation)

- Contract violations fail fast before backend submission.
- Unsupported target mappings fail with explicit errors.
- Missing storage/runtime configuration surfaces as explicit setup failure, not silent fallback.
- External scheduler failures (future live clients) should be represented as submitted/failed lifecycle states with immutable request IDs.

## Layer 4 — Execution flows

### A. Local happy path

1. Engineer submits immutable notebook request for local target.
2. Intake validates request and revision semantics.
3. Orchestrator resolves local backend and tracking URI.
4. Vertical slice executes training and logs MLflow run metadata.
5. Artifact and model-version records are generated.
6. Deployment record is emitted for local environment.
7. Prediction logs capture runtime metadata and traceability links.
8. Run visibility returned with MLflow deep link.

### B. Distributed submission path (Slurm/Kubernetes)

1. Engineer submits immutable notebook request for Lambda.ai/AWS target.
2. Intake validates request and revision semantics.
3. Orchestrator resolves backend and emits scheduler-specific payload.
4. Payload is submitted by future scheduler client adapters.
5. Status transitions move through submitted/running/finished/failed with stable request IDs.
6. Final run visibility should preserve the same traceability fields as local path.

### C. Failure/retry/rollback hooks

- Retry: same request lineage with explicit retry attempt metadata.
- Rollback: deployment record references predecessor version and rollback reason.
- Incident correlation: request ID + run ID + model version + deployment ID unify troubleshooting.

## Layer 5 — Operational coupling analysis

### Observability coupling

- MLflow run/state links are the primary operational pivot.
- Prediction logs bind inference behavior to deployment and model identity.
- Future scheduler integration must preserve this continuity for remote targets.

### Security coupling

- Immutable revisions reduce source drift risk between reviewed and executed code.
- Runtime credentials must remain injected and never persisted in notebooks.
- Environment boundaries (local, Lambda.ai, AWS) require distinct least-privilege credentials.

### Cost coupling

- Request and run metadata should include environment + workload identifiers for attribution.
- Slurm/Kubernetes clients should propagate cost tags at submission boundary.

### Deployment coupling

- Local and remote paths should expose the same promotion and rollback semantics.
- Infrastructure generation (Terranix/OpenTofu JSON) must stay aligned with runtime orchestration assumptions.

## Risk register and control points

| Risk | Impact | Control |
| --- | --- | --- |
| Divergent backend payload semantics between Slurm and Kubernetes | Non-portable behavior and failed promotions | Contract tests over normalized job spec and per-backend mapping fixtures |
| Traceability drift across environments | Incomplete audit and debugging gaps | Mandatory core lineage fields in all run/deploy/predict records |
| Config drift between flake and devenv infra generation | Non-reproducible infra behavior | Equivalence tests for Terranix-generated OpenTofu JSON outputs |
| Hidden secret handling in notebooks | Security exposure and non-auditable execution | Runtime-only secret injection contract, notebook immutability enforcement |
| Scheduler integration failures without clear lifecycle states | Operational blind spots | Explicit submission lifecycle model and failure taxonomy in contracts |


## Integration map (component interactions)

```mermaid
flowchart LR
  UI[Notebook Web UI] --> Intake[Notebook Intake Validation]
  Intake --> Orchestrator[Execution Orchestrator]
  Orchestrator --> Local[Local Vertical Slice Runtime]
  Orchestrator --> Slurm[Slurm Submission Payload]
  Orchestrator --> K8s[Kubernetes Submission Payload]
  Local --> MLflow[MLflow Tracking]
  Local --> Artifacts[Artifact + Deployment + Prediction Records]
  Terranix[Terranix via flake/devenv] --> OpenTofu[OpenTofu JSON Config]
  OpenTofu --> Infra[Provisioned Platform Surfaces]
  Infra --> Slurm
  Infra --> K8s
  MLflow --> UI
```


## Implementation-facing conclusions

1. The orchestration contract should remain the single canonical boundary between UI intent and backend execution semantics.
2. Scheduler integrations should be added as adapters under existing normalized job-spec contracts, not by introducing per-environment request models.
3. Infra generation must be testable as data artifacts (OpenTofu JSON), with parity checks across flake and devenv pipelines.
4. Traceability continuity is the highest-priority invariant across all execution paths.
5. Operational and governance controls should be modeled as first-class contract obligations, not post-hoc observability add-ons.